# Italian LLM Evaluation - Colab Quickstart

This notebook is a thin frontend for the repository. It keeps the evaluation logic in the package and only orchestrates setup, configuration, execution, and output inspection.

Use this for:
- a fast smoke test on a small public causal LM
- a quick run on a Hugging Face model ID
- validating that the environment works before larger evaluations


## What this notebook does

1. Clones your fork or the upstream repo
2. Installs the pinned runtime stack
3. Lets you choose a model source
4. Writes a temporary quick config
5. Runs tests
6. Runs the quick evaluation entry point
7. Shows where outputs were written


In [ ]:
# Edit these values before running if needed.

REPO_URL = "https://github.com/GiorgosPeikos/it_eval_autoregressive_llms.git"  # Replace with your fork when needed.
REPO_DIR = "it_eval_autoregressive_llms"

# Set to a public HF repo id for a quick smoke run.
# For local checkpoints on Drive, use a mounted path like /content/drive/MyDrive/models/your-checkpoint
MODEL_SOURCE = "Gpeik/Sophira-360M-base"
MODEL_REVISION = None  # Automatically resolve the model repository's current commit SHA.
TOKENIZER_SOURCE = "Gpeik/Sophira-360M-base"
TOKENIZER_REVISION = None  # Resolve independently when TOKENIZER_SOURCE is another repository.

# Keep this small for fast validation.
ENABLE_LIGHTEVAL = False
LIGHTEVAL_SUITE = "quick"  # quick, full, verified_windows, or all.
MAX_LIGHTEVAL_SAMPLES = 2
MAX_BLIMP_SAMPLES = 4
ENABLE_BLIMP_IT = True
ENABLE_PERPLEXITY = True
ENABLE_GENERATION = True
OVERWRITE_RESULTS = False
SAVE_DETAILS = True  # Disable for a smaller LightEval output bundle.

# Perplexity corpus and evaluation budget.
PPL_DATASET_REPO = "gsarti/clean_mc4_it"
PPL_DATASET_SUBSET = "tiny"  # clean_mc4_it: tiny=1/8, small=1/4, medium=1/2, large=3/4, full=all.
PPL_DATASET_SPLIT = "validation"
PPL_DATASET_STREAMING = True  # Avoid downloading/materializing unused splits.
MAX_PPL_DOCUMENTS = 3  # Set to None to score the complete selected subset.
MAX_PPL_TOKENS_PER_DOCUMENT = 256
MAX_GENERATION_PROMPTS = 3

# Leave as None to auto-select cuda when Colab exposes a GPU, otherwise cpu.
MODEL_DEVICE = None
PARALLELISM = "auto"  # Uses replicated inference when the runtime exposes multiple GPUs.
NUM_PROCESSES = "auto"  # Or set an integer no larger than the visible GPU count.

# Optional HF token for gated/private models.
HF_TOKEN = ""


In [ ]:
import os
import shutil
from pathlib import Path

%cd /content

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

repo_path = Path("/content") / REPO_DIR
if repo_path.exists():
    shutil.rmtree(repo_path)

!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/{REPO_DIR}

In [ ]:
!python --version

## Install the pinned stack

The repository targets Python 3.10 to 3.13 for the LightEval plus legacy-dataset path. Colab normally provides a compatible Python runtime.


In [ ]:
!python -m pip install --upgrade "pip<27" "setuptools<82" wheel
!python -m pip install "lighteval[multilingual]==0.13.0" --no-deps
!python -m pip install -r constraints/lighteval-python310-313.txt
!python -m pip install -e .[dev] --no-deps

In [ ]:
import torch
from it_eval_framework.utils.lighteval_runtime import lighteval_environment_report

if ENABLE_LIGHTEVAL:
    lighteval_report = lighteval_environment_report()
    print(f"LightEval preflight: {lighteval_report}")
    if lighteval_report["errors"]:
        raise RuntimeError("LightEval preflight failed; rerun the installation cell.")

detected_device = "cuda" if torch.cuda.is_available() else "cpu"
selected_device = MODEL_DEVICE or detected_device

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"selected device: {selected_device}")
if torch.cuda.is_available():
    print(f"cuda device count: {torch.cuda.device_count()}")
    print(f"cuda device name: {torch.cuda.get_device_name(0)}")
    print(f"cuda capability: {torch.cuda.get_device_capability(0)}")
else:
    print("No CUDA GPU is available in this runtime. In Colab, switch Runtime > Change runtime type > GPU.")


In [ ]:
from pathlib import Path
import yaml

config_path = Path("configs/colab_quickstart.yaml")
config_payload = {
    "run_name": "colab_quickstart",
    "model": {
        "source": MODEL_SOURCE,
        "revision": MODEL_REVISION,
        "tokenizer_source": TOKENIZER_SOURCE or MODEL_SOURCE,
        "tokenizer_revision": TOKENIZER_REVISION,
        "dtype": "auto",
        "device": selected_device,
        "batch_size": 1,
    },
    "output": {
        "root_dir": "evaluation_results",
        "overwrite": OVERWRITE_RESULTS,
        "save_details": SAVE_DETAILS,
    },
    "runtime": {
        "seed": 13,
        "python_executable": "python",
        "lighteval_command": "lighteval",
        "parallelism": PARALLELISM,
        "num_processes": NUM_PROCESSES,
    },
    "lighteval": {
        "enabled": ENABLE_LIGHTEVAL,
        "suite": LIGHTEVAL_SUITE if ENABLE_LIGHTEVAL else None,
        "max_samples": MAX_LIGHTEVAL_SAMPLES,
        "dataset_loading_processes": 1,
    },
    "blimp_it": {
        "enabled": ENABLE_BLIMP_IT,
        "dataset_revision": "4159ecb68388283488cb1d235a7e1946489bc62d",
        "max_samples": MAX_BLIMP_SAMPLES,
    },
    "perplexity": {
        "enabled": ENABLE_PERPLEXITY,
        "dataset_repo": PPL_DATASET_REPO,
        "dataset_subset": PPL_DATASET_SUBSET,
        "dataset_revision": "167d5696e91ac89f17936f9d0059031cbc4c9e99",
        "dataset_trust_remote_code": True,
        "dataset_streaming": PPL_DATASET_STREAMING,
        "split": PPL_DATASET_SPLIT,
        "text_field": "text",
        "sequence_length": 128,
        "stride": 64,
        "preserve_document_boundaries": True,
        "max_documents": MAX_PPL_DOCUMENTS,
        "max_tokens_per_document": MAX_PPL_TOKENS_PER_DOCUMENT,
    },
    "generation": {
        "enabled": ENABLE_GENERATION,
        "prompts_path": "configs/generation_prompts.yaml",
        "max_prompts": MAX_GENERATION_PROMPTS,
        "seed": 13,
    },
}

with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(config_payload, handle, allow_unicode=True, sort_keys=False)

print(config_path)
print(config_path.read_text(encoding="utf-8"))

In [ ]:
!python -m pytest

In [ ]:
!it-eval-launch --config configs/colab_quickstart.yaml

## Evaluation results

The next cell shows stage completion, metrics for every enabled evaluation, the rendered report, and a bounded preview of generated continuations.


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import pandas as pd

run_configs = list(Path("evaluation_results").rglob("run_config.yaml"))
if not run_configs:
    raise FileNotFoundError("No evaluation run was found. Run the evaluation cell first.")
run_dir = max(run_configs, key=lambda path: path.stat().st_mtime).parent
display(Markdown(f"**Complete result directory:** `{run_dir}`"))

state_path = run_dir / "run_state.json"
if state_path.exists():
    steps = json.loads(state_path.read_text(encoding="utf-8")).get("steps", {})
    status_rows = [{"stage": stage, **details} for stage, details in steps.items()]
    display(Markdown("### Stage status"))
    display(pd.DataFrame(status_rows).fillna("—"))

summary_path = run_dir / "summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"The run has no summary yet: {summary_path}")
summary = pd.read_csv(summary_path)
component_labels = {"blimp_it": "BLiMP-IT", "perplexity": "Perplexity", "generation": "Generation", "lighteval": "LightEval"}
for component, metrics in summary.groupby("component", sort=False):
    display(Markdown(f"### {component_labels.get(component, component)} metrics"))
    visible = metrics.drop(columns=["component"]).dropna(axis=1, how="all").reset_index(drop=True)
    display(visible)

report_path = run_dir / "report.md"
if report_path.exists():
    display(Markdown("### Complete metric report"))
    display(Markdown(report_path.read_text(encoding="utf-8")))

generations_path = run_dir / "generations.jsonl"
if generations_path.exists():
    generation_rows = [json.loads(line) for line in generations_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    preview = pd.DataFrame([{
        "prompt_id": row.get("prompt_identifier"),
        "profile": row.get("decoding_profile", {}).get("name"),
        "prompt": row.get("prompt_text"),
        "generated_text": row.get("generated_text", "")[:500],
    } for row in generation_rows[:6]])
    display(Markdown(f"### Generation preview ({len(preview)} of {len(generation_rows)})"))
    with pd.option_context("display.max_colwidth", 500):
        display(preview)

## Next steps after the quick run

- Replace `MODEL_SOURCE` and `TOKENIZER_SOURCE` if you want to evaluate a model other than Sophira-360M
- Increase sample limits gradually
- Switch from `configs/colab_quickstart.yaml` to a full config
- The default perplexity example uses `gsarti/clean_mc4_it` (Italian only); for publication-grade evaluation, replace it with your own Italian held-out file if needed
